# create-splat — production training

**Project:** `{{PROJECT_NAME}}`  
**Trainer:** nerfstudio splatfacto (Apache-2.0). Commercial use OK.

## What to do

1. `Runtime → Change runtime type → GPU` (T4 is fine — it's the free default).
2. `Runtime → Run all`.
3. Close the tab if you want. The result lands back in your Google Drive.

Expected wall time on a T4: **25–40 min** for 80–200 photos at 30k iterations.

If anything fails, the cell prints the tail of the failing step's log automatically. Paste that into cowork and the next iteration of the skill will target the exact failure.

In [ ]:
# Single-cell production run — install + SfM + train + export + write back to Drive.
#
# Why one cell, no kernel restart: we install nerfstudio into a venv created with
# --system-site-packages, which inherits Colab's preinstalled torch+CUDA but lets
# us pin numpy<2 / protobuf<4 locally without breaking the running kernel. All
# nerfstudio CLI invocations go through the venv's ns-* binaries via subprocess,
# so the host kernel never imports nerfstudio at all — no in-memory ABI conflict,
# no kill-and-restart required.
#
# Headless-Linux fixes baked in (discovered the hard way across four debug rounds):
#   • QT_QPA_PLATFORM=offscreen — Ubuntu's colmap links Qt5 GUI libs.
#   • --no-gpu on ns-process-data — colmap's GPU SIFT needs OpenGL/EGL, Colab has
#     CUDA but no GL display. CPU SIFT works fine, only slightly slower.
#   • --viewer.quit-on-train-completion True — current tyro on Colab requires an
#     explicit True/False value (a bare flag is parsed as missing-value and the
#     *next* argv token, e.g. --vis, gets consumed as the value, causing a parse
#     error: "invalid choice '--vis' … expected one of ('True', 'False')").
#   • Export step patches torch.load(weights_only=False) and calls nerfstudio's
#     exporter entrypoint() in-process from a standalone script run with the
#     venv's own python3 — PyTorch 2.6 flipped torch.load's default to
#     weights_only=True, which breaks nerfstudio's own checkpoint loading, and
#     a sitecustomize.py patch does NOT get picked up inside a
#     --system-site-packages venv, so the patch must be applied directly inside
#     the same process that imports nerfstudio (a fresh `ns-export` subprocess
#     would re-import a clean, unpatched torch every time).
#
# Output: writes splat.ply, transforms.json, and a .done marker to
#   /content/drive/MyDrive/splats/{{PROJECT_NAME}}/
# The local create-splat skill watches for .done and continues from there.

import os, sys, subprocess, time, shutil, glob, json
from google.colab import drive
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_COMPILE_DISABLE'] = '1'

PROJECT = '6997d9bc19d1'  # ← CHANGE THIS to your actual project ID before running
ITERATIONS = 30000  # bump to 60000 for higher quality (will need ~2× wall time)

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = f'/content/drive/MyDrive/splats/{PROJECT}'
SRC = f'{DRIVE_ROOT}/photos'
OUT = DRIVE_ROOT
assert os.path.isdir(SRC), f'✗ no photos at {SRC} — cowork should have written them here'
n_photos = len([f for f in os.listdir(SRC) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
assert n_photos >= 20, f'✗ only {n_photos} photos at {SRC}'
print(f'project={PROJECT!r}  photos={n_photos}  iters={ITERATIONS}')

# Headless Qt for any subprocess we spawn.
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

def step(label, args, log_path):
    print(f'\n=== {label} ===')
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=os.environ)
    with open(log_path, 'w') as log:
        for line in p.stdout:
            print(line, end='')
            log.write(line)
    p.wait()
    dt = time.time() - t0
    print(f'\nrc={p.returncode}  ({dt:.0f}s)')
    if p.returncode != 0:
        print(f'\n—— last 60 lines of {log_path} ——')
        subprocess.run(['tail', '-60', log_path])
        raise RuntimeError(f'✗ {label} failed (rc={p.returncode})')

# 1. COLMAP system binary (fast).
step('install colmap',
     ['bash', '-c', 'apt-get update -qq && apt-get install -qq -y colmap'],
     '/content/colmap_install.log')

VENV = '/content/venv'
step('create venv',
     [sys.executable, '-m', 'venv', VENV, '--system-site-packages', '--without-pip'],
     '/content/venv.log')
step('bootstrap pip in venv',
     ['bash', '-c',
      f'curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py && {VENV}/bin/python3 /tmp/get-pip.py'],
     '/content/getpip.log')

# 3. Install nerfstudio + pinned deps inside the venv. The venv's pip will pull
# numpy<2 / protobuf<4 in front of the system versions for any process started
# from the venv. The running kernel keeps its own numpy and never touches
# nerfstudio — so no restart is needed.
step('upgrade venv pip',
     [f'{VENV}/bin/pip', 'install', '--quiet', '--upgrade', 'pip'],
     '/content/pip_upgrade.log')
step('pin numpy<2 protobuf<4 in venv',
     [f'{VENV}/bin/pip', 'install', '--quiet', 'numpy<2', 'protobuf<4'],
     '/content/pin.log')
step('install nerfstudio in venv (slow: 3–7 min)',
     [f'{VENV}/bin/pip', 'install', '--quiet', 'nerfstudio'],
     '/content/nerfstudio_install.log')
step('install ninja (system binary, on PATH)',
     ['apt-get', 'install', '-y', 'ninja-build'],
     '/content/ninja_install.log')

# 4. SfM with CPU SIFT (avoids OpenGL requirement of GPU SIFT on headless Colab).
shutil.rmtree('/content/processed', ignore_errors=True)
# Sequential matching (rather than exhaustive) is the right choice for one
# continuous video walkthrough: it compares each frame mainly to its near
# neighbors in time, matching how the footage was actually captured, instead
# of comparing every possible pair (which is slower and wastes effort on
# frame pairs that share no real overlap, e.g. entryway vs. kitchen).
step('SfM (ns-process-data, ~3–8 min)',
     [f'{VENV}/bin/ns-process-data', 'images',
      '--data', SRC,
      '--output-dir', '/content/processed',
      '--matching-method', 'sequential',
      '--no-gpu'],
     '/content/sfm.log')

# Sanity check: transforms.json should have most frames registered.
tj = json.load(open('/content/processed/transforms.json'))
n_registered = len(tj.get('frames', []))
print(f'\nSfM registered {n_registered}/{n_photos} frames')
assert n_registered >= 10, '✗ SfM registered too few frames — capture coverage likely poor'

# 5. Train splatfacto.
shutil.rmtree('/content/runs', ignore_errors=True)
# Quality-tuned splatfacto flags, recommended by nerfstudio's own docs for
# scenes where quality matters more than training speed or file size —
# directly applicable here given the low-texture white walls/counters in
# Gawain's rooms (matches the exact symptom in nerfstudio GitHub issue #3393:
# "good in some parts, lots of noise in the air" on a similar low-feature room):
#   • cull_alpha_thresh=0.005 (default 0.1) — keep more faint-but-real detail
#     instead of deleting it as noise.
#   • continue_cull_post_densification=False — stop aggressively culling after
#     the densification phase (15k steps), preserving more of what's been built.
#   • use_scale_regularization=True — PhysGaussian-style regularizer that
#     discourages long, spikey/streaky Gaussians, a major contributor to the
#     "messy" look.
step(f'train splatfacto ({ITERATIONS} iters, ~25–40 min on T4)',
     [f'{VENV}/bin/ns-train', 'splatfacto',
      '--data', '/content/processed',
      '--output-dir', '/content/runs',
      '--max-num-iterations', str(ITERATIONS),
      '--pipeline.model.cull-alpha-thresh', '0.005',
      '--pipeline.model.continue-cull-post-densification', 'False',
      '--pipeline.model.use-scale-regularization', 'True',
      '--viewer.quit-on-train-completion', 'True',
      '--vis', 'tensorboard'],
     '/content/train.log')

# 6. Find the run config + verify checkpoints exist.
configs = sorted(glob.glob('/content/runs/**/config.yml', recursive=True), key=os.path.getmtime)
assert configs, '✗ no config.yml — training did not start'
CONFIG = configs[-1]
ckpt_dir = os.path.join(os.path.dirname(CONFIG), 'nerfstudio_models')
ckpts = glob.glob(f'{ckpt_dir}/*')
assert ckpts, f'✗ no checkpoints in {ckpt_dir} — training crashed before first save'
print(f'\n✓ training produced {len(ckpts)} checkpoint(s); using config {CONFIG}')

# 7. Export to .ply.
#
# PyTorch 2.6 changed torch.load's default to weights_only=True, which breaks
# nerfstudio's own checkpoint loading (_pickle.UnpicklingError on
# numpy.core.multiarray.scalar). A sitecustomize.py patch does NOT get picked
# up inside a --system-site-packages venv, so instead we write a standalone
# script that patches torch.load BEFORE importing nerfstudio, then calls the
# exporter's entrypoint() directly in-process — and we run that script with
# the venv's own python3 (not the notebook's kernel, which doesn't have
# nerfstudio installed at all, and not a fresh `ns-export` subprocess, which
# would re-import a clean, unpatched torch).
os.makedirs('/content/export', exist_ok=True)

_export_script = f'''
import sys, os
import torch
_orig_load = torch.load
def _patched_load(*a, **kw):
    kw["weights_only"] = False
    return _orig_load(*a, **kw)
torch.load = _patched_load

sys.argv = [
    "ns-export", "gaussian-splat",
    "--load-config", "{CONFIG}",
    "--output-dir", "/content/export",
]
os.makedirs("/content/export", exist_ok=True)
os.environ["QT_QPA_PLATFORM"] = "offscreen"

from nerfstudio.scripts.exporter import entrypoint
entrypoint()
'''
with open('/content/export_patched.py', 'w') as f:
    f.write(_export_script)

step('export gaussian-splat',
     [f'{VENV}/bin/python3', '/content/export_patched.py'],
     '/content/export.log')
plys = sorted(glob.glob('/content/export/**/*.ply', recursive=True), key=os.path.getmtime)
assert plys, '✗ no .ply in /content/export'
PLY = plys[-1]
print(f'exported {PLY}  ({os.path.getsize(PLY)/1e6:.1f} MB)')

# 8. Write outputs back to Drive — splat.ply, transforms.json, and a .done marker.
# The local create-splat skill polls the .done file every 60s and picks up from there.
os.makedirs(OUT, exist_ok=True)
shutil.copy(PLY, f'{OUT}/splat.ply')
shutil.copy('/content/processed/transforms.json', f'{OUT}/transforms.json')
with open(f'{OUT}/.done', 'w') as f:
    f.write(json.dumps({
        'project': PROJECT,
        'iterations': ITERATIONS,
        'n_photos': n_photos,
        'n_registered': n_registered,
        'ply_size_mb': round(os.path.getsize(f'{OUT}/splat.ply') / 1e6, 1),
        'ckpts': len(ckpts),
    }))
print(f'\n✓ ALL DONE — splat.ply + transforms.json + .done written to {OUT}')
print('  cowork is polling for .done; you can close this tab.')

# 9. Launch a live 3D viewer of the trained scene and print a clickable link.
#
# Same weights_only problem as step 7 hits ns-viewer too (it calls the same
# eval_load_checkpoint -> torch.load path), so we patch it the same way: a
# standalone script run with the venv's python3, never the bare ns-viewer
# subprocess. The viewer is a long-lived server (viser, listening on
# 0.0.0.0:7007 inside the VM), so we start it in the background and use
# Colab's own proxyPort helper to get a public-ish URL that tunnels into it.
print('\n=== starting live viewer ===')

_viewer_script = f'''
import sys, os
import torch
_orig_load = torch.load
def _patched_load(*a, **kw):
    kw["weights_only"] = False
    return _orig_load(*a, **kw)
torch.load = _patched_load

sys.argv = [
    "ns-viewer",
    "--load-config", "{CONFIG}",
]
os.environ["QT_QPA_PLATFORM"] = "offscreen"

from nerfstudio.scripts.viewer.run_viewer import entrypoint
entrypoint()
'''
with open('/content/viewer_patched.py', 'w') as f:
    f.write(_viewer_script)

viewer_proc = subprocess.Popen(
    [f'{VENV}/bin/python3', '/content/viewer_patched.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

# Wait until the viewer has actually finished its startup sequence — the
# viser box printing only means the HTTP server started listening, NOT that
# the viewer is ready to serve a working page. After the box, nerfstudio still
# has to cache/undistort eval images, which scales with frame count and can
# take well over a minute on larger videos. Grabbing the proxy URL too early
# (right after the viser box) returns a URL that loads but 500s, since the
# server isn't fully initialized yet. So we wait for the actual completion
# marker instead — "disabled local writer", which nerfstudio prints right
# after eval-image caching finishes — with a generous 10-minute ceiling and a
# fixed settle buffer afterward, rather than a short fixed timeout.
_viewer_ready = False
_t0 = time.time()
while time.time() - _t0 < 600:
    line = viewer_proc.stdout.readline()
    if line:
        print(line, end='')
        if 'disabled local writer' in line.lower():
            _viewer_ready = True
            break
    if viewer_proc.poll() is not None:
        break
    if not line:
        time.sleep(0.5)

if viewer_proc.poll() is not None:
    print(f'\n✗ viewer process exited early (rc={viewer_proc.returncode}) — check output above')
elif not _viewer_ready:
    print('\n✗ viewer did not reach ready state within 10 minutes — check output above, it may still be starting')
else:
    time.sleep(5)  # small settle buffer so the server is fully serving requests
    from google.colab.output import eval_js
    viewer_url = eval_js("google.colab.kernel.proxyPort(7007)")
    print(f'\n✓ VIEWER LIVE — open this link to see the 3D splat: {viewer_url}')
    print('  (viewer keeps running in the background; re-run eval_js("google.colab.kernel.proxyPort(7007)") any time to get the link again)')
